In [2]:
import sys, os
import numpy as np
sys.path.append(os.getcwd())
from src.autoencoder.autoencoder_module import PointNetAutoencoder
from src.autoencoder.eval_autoencoder import create_autoencoder_dataloader, get_batch_reconstructions
import numpy as np
import matplotlib.pyplot as plt
from omegaconf import DictConfig, OmegaConf
from hydra import compose, initialize

from warnings import filterwarnings
filterwarnings("ignore")

import torch
from src.utils.model_utils import load_model_from_checkpoint

In [3]:
with initialize(version_base=None, config_path="configs"):
    cfg = compose(config_name="autoencoder")

# checkpoit_path = 'output/2025-02-23/19-44-33/pointnet-epoch=3079-val_loss=0.05.ckpt'
checkpoint_path = 'output/2025-03-04/18-43-36/pointnet-epoch=749-val_loss=0.04.ckpt'
cuda_device = 0
device = f'cuda:{cuda_device}' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {device}")
model = load_model_from_checkpoint(checkpoint_path, cfg, device=device)


    



Using device: cuda:0


PointNetAE_Transformer_Folding
Loss: chamfer_kl_divergence_loss


Model loaded successfully from output/2025-03-04/18-43-36/pointnet-epoch=749-val_loss=0.04.ckpt
Model type: PointNetAE_Transformer_Folding
Latent size: 8


In [4]:
file_path_crystals = 'datasets/Al/inherent_configurations_off/240ps.off'
dataloader_crystals = create_autoencoder_dataloader(cfg, file_path_crystals)
points_crystals = next(iter(dataloader_crystals))[0]
original_points_c, reconstructed_points_c = get_batch_reconstructions(
    model, points_crystals, cfg.data.num_points, device=f'cuda:{cuda_device}'
)


file_path_liquid = 'datasets/Al/inherent_configurations_off/166ps.off'
dataloader_liquid = create_autoencoder_dataloader(cfg, file_path_liquid)
points_liquid = next(iter(dataloader_liquid))[0]
original_points_l, reconstructed_points_l = get_batch_reconstructions(
    model, points_liquid, cfg.data.num_points, device=f'cuda:{cuda_device}'
)

Loading cached file from datasets/Al/inherent_configurations_off/240ps.npy


Avg added 0.01 points, avg dropped 14.11 points


Number of samples in spheric dataset: 46656
Input points shape: torch.Size([2048, 64, 3])
Reconstructed shape: torch.Size([2048, 64, 3])
Original points shape after processing: (2048, 64, 4)
Reconstructed points shape after processing: (2048, 64, 4)
Loading cached file from datasets/Al/inherent_configurations_off/166ps.npy


Avg added 1.35 points, avg dropped 10.1 points


Number of samples in spheric dataset: 50653
Input points shape: torch.Size([2048, 64, 3])
Reconstructed shape: torch.Size([2048, 64, 3])
Original points shape after processing: (2048, 64, 4)
Reconstructed points shape after processing: (2048, 64, 4)


In [5]:
from src.eval_tools.vis_utils import visualize_original_and_reconstructed

# visualize_original_and_reconstructed(original_points_c, reconstructed_points_c, random_sample=True)

In [6]:
from scipy.spatial.distance import cdist


original_points = original_points_c
reconstructed_points = reconstructed_points_c

In [7]:
def compute_pointcloud_distances(original_points, reconstructed_points, metric='euclidean'):
    """
    Compute distances between N pairs of original and reconstructed point clouds.
    
    Parameters:
    -----------
    original_points: numpy.ndarray
        Array of original point clouds with shape (N, num_points, dim)
    reconstructed_points: numpy.ndarray
        Array of reconstructed point clouds with shape (N, num_points, dim)
    metric: str, optional
        Distance metric to use for cdist (default: 'euclidean')
        
    Returns:
    --------
    all_distances: numpy.ndarray
        Array of shape (N,) containing the average distance for each point cloud pair
    point_distances: numpy.ndarray
        Array of shape (N, num_points) containing distances for each point in each cloud
    """
    N = original_points.shape[0]
    num_points = original_points.shape[1]
    
    all_distances = np.zeros(N)
    point_distances = np.zeros((N, num_points))
    
    for i in range(N):
        # Compute distance matrix between original and reconstructed points
        dist_matrix = cdist(original_points[i], reconstructed_points[i], metric=metric)
        
        # For each point in original, find the minimum distance to any point in reconstructed
        min_distances = np.min(dist_matrix, axis=1)
        
        # Store distances
        point_distances[i] = min_distances
        all_distances[i] = np.mean(min_distances)
    
    return all_distances, point_distances

# Calculate distances
# avg_distances, per_point_distances = compute_pointcloud_distances(original_points, reconstructed_points)

# Print statistics
# print(f"Average distance across all pairs: {np.mean(avg_distances):.4f}")
# print(f"Min distance: {np.min(avg_distances):.4f}, Max distance: {np.max(avg_distances):.4f}")

def print_distance_statistics(all_distances, point_distances):
    """
    Print statistical measures for point cloud distances.
    
    Parameters:
    -----------
    all_distances: numpy.ndarray
        Array of shape (N,) containing the average distance for each point cloud pair
    point_distances: numpy.ndarray
        Array of shape (N, num_points) containing distances for each point in each cloud
    """
    
    print("\n Distance Statistics")
    print(f"Cloud-level statistics (across {len(all_distances)} point cloud pairs):")
    print(f"  Mean distance:       {np.mean(all_distances):.6f}")
    print(f"  Standard deviation:  {np.std(all_distances):.6f}")
    print(f"  Min distance:        {np.min(all_distances):.6f}")
    print(f"  Max distance:        {np.max(all_distances):.6f}")
    print(f"  Median distance:     {np.median(all_distances):.6f}")
    
    # all_point_distances = point_distances.flatten()
    # print(f"\nPoint-level statistics (across {len(all_point_distances)} points):")
    # print(f"  Mean distance:       {np.mean(all_point_distances):.6f}")
    # print(f"  Standard deviation:  {np.std(all_point_distances):.6f}")
    # print(f"  Min distance:        {np.min(all_point_distances):.6f}")
    # print(f"  Max distance:        {np.max(all_point_distances):.6f}")
    # print(f"  Median distance:     {np.median(all_point_distances):.6f}")
    
    # percentiles = [5, 25, 75, 95]
    # perc_values = np.percentile(all_point_distances, percentiles)
    # print("\nPercentiles of point-level distances:")
    # for p, v in zip(percentiles, perc_values):
    #     print(f"  {p}th percentile:     {v:.6f}")


In [8]:
# metric = 'hamming'
avg_distances, per_point_distances = compute_pointcloud_distances(original_points, reconstructed_points, metric='hamming')
print_distance_statistics(avg_distances, per_point_distances)

# plt.figure(figsize=(10, 6))
# plt.hist(avg_distances, bins=50, alpha=0.7, color='blue')
# plt.xlabel('Average Distance')
# plt.ylabel('Frequency')
# plt.title('Distribution of Average Distances Between Original and Reconstructed Point Clouds')
# plt.grid(alpha=0.3)
# plt.show()


 Distance Statistics
Cloud-level statistics (across 2048 point cloud pairs):
  Mean distance:       0.750000
  Standard deviation:  0.000000
  Min distance:        0.750000
  Max distance:        0.750000
  Median distance:     0.750000
